# 0a. Real Data Explorer

Load experimental data, browse animals and sessions, visualise behaviour.

**Sections:**
1. Load data
2. Class reference — what each class stores and how to use it
3. Experiment overview
4. Animal trajectory (across sessions)
5. Single-session visualisation
6. Summary stats table
7. Scratch space


In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================

import sys
import os

sys.path.append(os.path.abspath(".."))

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore')

from Data.structures import (
    load_session_csv, load_animal, load_experiment,
    AnimalData, ExperimentData, SessionData, TrialData,
    stimulus_to_category,
)
from Analysis.summary_stats import (
    compute_summary_stats,
    list_available_stats,
    get_stat_names_expanded,
)
from Analysis.session_features import build_feature_matrix, build_feature_matrix_multi
from Plotting.psychometric import plot_psychometric, plot_session_psychometrics
from Plotting.session import plot_session_trials

ALL_STATS = list_available_stats()
print('Setup complete.')


---

## 1. Load Data

Uncomment whichever option matches your setup. All three produce an `ExperimentData` object.

In [ ]:
# =============================================================================
# DATA LOADING — pick one option
# =============================================================================

STAGE = 'Full_Task_Cont'  # stage filter (None = all)

# --- Option 1: standard directory tree ---
path = '/Users/Serkan/Desktop/pro/PhD/main/data/Head_Fixed_Behavior/Processed'
experiment = load_experiment(Path(path))

# --- Option 2: per-animal directory ---
# experiment = ExperimentData()
# for aid, path in [('SS05', '/path/to/SS05'), ('SS06', '/path/to/SS06')]:
#     experiment.add_animal(load_animal(aid, path))

# --- Option 3: explicit CSV paths ---
# csv_paths = {
#     'SS05': sorted(Path('/Head_Fixed_Behavior/Processed/SS05').rglob('trial_summary_*.csv')),
#     # 'SS06': sorted(Path('/path/to/SS06').rglob('trial_summary_*.csv')),
# }
# experiment = ExperimentData()
# for aid, paths in csv_paths.items():
#     sessions = []
#     for idx, p in enumerate(paths):
#         try:
#             sess = load_session_csv(p, session_idx=idx)
#             sessions.append(sess)
#         except Exception as e:
#             print(f'  Skip {p.name}: {e}')
#     if sessions:
#         experiment.add_animal(AnimalData(animal_id=aid, sessions=sessions))
#         print(f'{aid}: {len(sessions)} sessions')

print(f'\nLoaded {experiment.n_animals} animals: {experiment.animal_ids}')


---

## 2. Class Reference

The data pipeline is built on a hierarchy of dataclasses:

```
ExperimentData                    	# all animals
  └── AnimalData                  	# one animal, all sessions
        └── SessionData           	# one session
              ├── SessionMetadata  	# task parameters (constant within session)
              ├── BlockInfo[]      	# distribution blocks
              └── TrialData        	# trial-by-trial arrays
					
	└── FittingData          	# model-ready extraction from AnimalData
  	      └── session_arrays[]	# list of dicts with 'stimuli', 'choices', etc.
```

Run the cells below to see what each class stores and how to use it.
These cells assume data has been loaded (Section 1).

### ExperimentData

Top-level container. Dict of animals keyed by ID.

In [ ]:
# --- ExperimentData ---
# After loading (Section 1), 'experiment' is an ExperimentData

print(type(experiment))
print(f'Animals: {experiment.animal_ids}')       # list of IDs
print(f'Count:   {experiment.n_animals}')         # number of animals

# Filter to animals with enough sessions
n_session_filter = 10
good_animals = experiment.get_animals_with_min_sessions(n_session_filter, stage='Full_Task_Cont')
print(f'Animals with >= {n_session_filter} task sessions: {[a.animal_id for a in good_animals]}')

# Summary table
display(experiment.summary())

# Save / load
# experiment.save('experiment.pkl')
# experiment = ExperimentData.load('experiment.pkl')


### AnimalData

One animal. Contains a chronological list of `SessionData`. The main interface
for filtering sessions and extracting model-ready data.

In [ ]:
# --- AnimalData ---

# Get one animal
animal = experiment.get_animal(experiment.animal_ids[0])

print(f'Animal: {animal.animal_id}')
print(f'Total sessions: {animal.n_sessions}')
print(f'Stages present: {animal.stages}')

# Filter sessions
task_sessions = animal.get_sessions(stage='Full_Task_Cont')  # by stage
# animal.get_sessions(distribution='Uniform')                # by distribution
# animal.get_sessions(idx_range=(0, 5))                      # by index range
# animal.get_sessions(date_range=(date(2026,1,1), date(2026,2,1)))  # by date
print(f'Full_Task_Cont sessions: {len(task_sessions)}')

# Summary table (one row per session)
display(animal.summary().head())


### FittingData (model-ready extraction)

`get_fitting_data()` filters sessions and trials, returning arrays ready for
the BE model / SBI pipeline. This is the bridge between raw data and modelling.

In [ ]:
# --- FittingData ---

fitting = animal.get_fitting_data(
    stage='Full_Task_Cont',
    exclude_abort=True,
    exclude_opto=False,
    exclude_no_response=False,
    time_axis='session_idx',   # or 'calendar_days'
    min_valid_trials=10,
)

print(f'Sessions: {fitting.n_sessions}')
print(f'Trials per session: {fitting.trials_per_session}')
print(f'Valid trials per session: {fitting.valid_trials_per_session}')
print(f'Time axis ({fitting.time_axis_type}): {fitting.time_axis}')

# Access per-session arrays
session_idx = 0
sa = fitting.get_session(session_idx)  # dict for given session index

# or 
sa_stimuli = fitting.stimuli[session_idx] # np array for given sesion index
sa_choices = fitting.choices[session_idx]
sa_categories = fitting.categories[session_idx]

print(f'\nSession 0 arrays: {list(sa.keys())}')
print(f'  stimuli:    shape={sa["stimuli"].shape}, range=[{sa["stimuli"].min():.2f}, {sa["stimuli"].max():.2f}]')
print(f'  choices:    shape={sa["choices"].shape}, unique={np.unique(sa["choices"][~np.isnan(sa["choices"])])}')
print(f'  categories: shape={sa["categories"].shape}')
print(f'  no_response: {sa["no_response"].sum()} trials')



### SessionData

One session. Contains metadata, block structure, and trial-level data.

In [ ]:
# --- SessionData ---

sess = task_sessions[0]

print(f'Session ID:    {sess.session_id}')
print(f'Date:          {sess.date}')
print(f'Session index: {sess.session_idx}')
print(f'Stage:         {sess.stage}')              # shortcut for metadata.stage
print(f'Distribution:  {sess.distribution}')       # shortcut for first block
print(f'N trials:      {sess.n_trials}')
print(f'N blocks:      {sess.n_blocks}')
print(f'CSV path:      {sess.csv_path}')

### SessionMetadata

Task parameters that are constant within a session (protocol, contingency, timing, etc.).

In [ ]:
# --- SessionMetadata ---

meta = sess.metadata

print(f'Protocol:           {meta.protocol}')
print(f'Stage:              {meta.stage}')
print(f'Sound contingency:  {meta.sound_contingency}')
print(f'Stim range:         [{meta.stim_range_min}, {meta.stim_range_max}]')
print(f'Response window:    {meta.response_window} ms')
print(f'Sound duration:     {meta.sound_duration} ms')
print(f'ITI:                {meta.iti} ms')
print(f'Anti-bias:          {meta.anti_bias}')


### BlockInfo

Distribution blocks within a session. Each block has a distribution type,
exponential rate parameter, and trial range.

In [ ]:
# --- BlockInfo ---

for block in sess.blocks:
    print(f'Block {block.block_idx}: {block.distribution} '
          f'(exp_rate={block.exp_rate}), '
          f'trials {block.trial_start}-{block.trial_end} '
          f'({block.n_trials} trials)')


### TrialData

Trial-by-trial arrays. This is where the actual behavioural data lives.

**Encoding conventions:**
- `stimulus`: relative value in [-1, 1], boundary at 0
- `category`: 0 = A (stimulus < 0), 1 = B (stimulus > 0)
- `choice_spatial`: -1 = left, 0 = no response, 1 = right
- `choice_category`: 0 = chose A, 1 = chose B, NaN = no response
- `correct`: True/False (from Bonsai, based on choice vs category)
- `abort`: True if animal broke fixation
- `no_response`: True if choice_category is NaN
- `valid_mask`: ~abort & ~no_response

In [ ]:
# --- TrialData ---

trials = sess.trials

print(f'Total trials:     {trials.n_trials}')
print(f'Valid trials:     {trials.valid_mask.sum()}')
print(f'No-response:      {trials.no_response.sum()}')
print(f'Aborts:           {trials.abort.sum()}')

# Core arrays
print(f'\nStimulus range:   [{trials.stimulus.min():.3f}, {trials.stimulus.max():.3f}]')
print(f'Categories:       {np.unique(trials.category)}')
print(f'Choices (spatial): {np.unique(trials.choice_spatial)}')
print(f'Choices (cat):     {np.unique(trials.choice_category[~np.isnan(trials.choice_category)])}')

# Opto
if trials.opto_on.any():
    print(f'\nOpto trials:      {trials.opto_on.sum()}')
else:
    print(f'\nNo opto trials in this session.')

# RT
valid_rt = trials.reaction_time[trials.valid_mask]
valid_rt = valid_rt[~np.isnan(valid_rt) & (valid_rt > 0)]
if len(valid_rt) > 0:
    print(f'RT: median={np.median(valid_rt):.0f} ms, '
          f'range=[{valid_rt.min():.0f}, {valid_rt.max():.0f}]')


### TrialData.get_model_arrays()

The main extraction method. Filters trials (aborts, opto) and returns
a dict of arrays ready for the BE model or summary stats.

This is what `FittingData.session_arrays` contains for each session.

In [ ]:
# --- get_model_arrays ---

arrays = trials.get_model_arrays(
    exclude_abort=True,         # remove abort trials
    exclude_opto=True,          # remove opto-on trials
    exclude_no_response=False,  # keep no-response (marked, not removed)
)

print('Keys:', list(arrays.keys()))
print(f'  stimuli:       {arrays["stimuli"].shape}  (stimulus values)')
print(f'  categories:    {arrays["categories"].shape}  (true category 0/1)')
print(f'  choices:       {arrays["choices"].shape}  (choice 0/1/NaN)')
print(f'  no_response:   {arrays["no_response"].sum()} NaN trials (kept but flagged)')
print(f'  not_blockstart: {(~arrays["not_blockstart"]).sum()} block boundaries')
print(f'  reaction_times: {arrays["reaction_times"].shape}')
print(f'  trial_indices:  original indices for back-mapping to TrialData')

# Typical usage: filter no-response then compute stats
valid = ~arrays['no_response']
stim = arrays['stimuli'][valid]
choices = arrays['choices'][valid]
cats = arrays['categories'][valid]
print(f'\nAfter no-response filter: {len(stim)} trials')


---

## 3. Experiment Overview

In [ ]:
# =============================================================================
# Summary table
# =============================================================================

rows = []
for aid in experiment.animal_ids:
    animal = experiment.get_animal(aid)
    sessions = animal.get_sessions(stage=STAGE) if STAGE else animal.sessions
    if not sessions:
        continue
    trial_counts = [s.trials.n_trials for s in sessions]
    valid_counts = [s.trials.valid_mask.sum() for s in sessions]
    rows.append({
        'animal': aid,
        'n_sessions': len(sessions),
        'date_first': sessions[0].date,
        'date_last': sessions[-1].date,
        'total_trials': sum(trial_counts),
        'mean_trials': int(np.mean(trial_counts)),
        'mean_valid': int(np.mean(valid_counts)),
        'stages': list(set(s.metadata.stage for s in sessions)),
    })

overview_df = pd.DataFrame(rows)
display(overview_df)


---

## 4. Animal Trajectory

Pick an animal to see key metrics across sessions.

In [ ]:
# =============================================================================
# Pick animal
# =============================================================================

ANIMAL_ID = experiment.animal_ids[6]

animal = experiment.get_animal(ANIMAL_ID)
sessions = animal.get_sessions(stage=STAGE) if STAGE else animal.sessions
print(f'{ANIMAL_ID}: {len(sessions)} sessions')


In [ ]:
# =============================================================================
# Build feature matrix (all summary stats + RT features across sessions)
# =============================================================================

feat_df = build_feature_matrix(
    animal, 
    stage=STAGE, 
    exclude_abort=False, 
    min_valid_trials=10
)

print(f'Feature matrix: {feat_df.shape[0]} sessions x {feat_df.shape[1]} columns')
print(f'Metadata columns: {[c for c in feat_df.columns if feat_df[c].dtype == object or c in ["session_idx", "date"]]}')

# Show first few rows
display(feat_df.head())


In [ ]:
# =============================================================================
# Learning trajectory: key metrics across sessions
# =============================================================================

metrics = ['accuracy', 'pse', 'slope', 'lapse_low', 'lapse_high', 'recency', 'side_bias']
available = [m for m in metrics if m in feat_df.columns]

n_cols = 3
n_rows = int(np.ceil(len(available) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows), sharex=True)
axes = axes.flatten()

x = feat_df['session_idx'].values

for i, metric in enumerate(available):
    ax = axes[i]
    vals = feat_df[metric].values
    ax.plot(x, vals, 'o-', markersize=4, linewidth=1, color='steelblue')
    ax.set_ylabel(metric)
    ax.axhline(0, color='gray', linewidth=0.5, alpha=0.5)
    ax.set_title(f'{metric}')
    
    
    # Highlight reference values
    if metric == 'accuracy':
        ax.axhline(0.5, color='red', linewidth=0.5, linestyle='--', alpha=0.5)
    if metric == 'pse':
        ax.axhline(0, color='red', linewidth=0.5, linestyle='--', alpha=0.5)

for i in range(len(available), len(axes)):
    axes[i].set_visible(False)

axes[-n_cols].set_xlabel('Session index')
fig.suptitle(f'{ANIMAL_ID} — learning trajectory', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# Psychometric curve evolution (grid of sessions)
# =============================================================================

idx_range = (5,10)
subset = animal.get_sessions(stage='Full_Task_Cont', idx_range=idx_range)

# all sessions
subset = sessions

fig, infos = plot_session_psychometrics(
    subset, 
    mode='grid',
    suptitle=f'{ANIMAL_ID} - Sessions {idx_range[0]}-{idx_range[1]}')

In [ ]:
# =============================================================================
# Pooled psychometric (all sessions combined, with bootstrap CIs)
# =============================================================================

idx_range = (5,10)
subset = animal.get_sessions(stage='Full_Task_Cont', idx_range=idx_range)

# all sessions
subset = sessions

fig, infos = plot_session_psychometrics(
    subset, 
    mode='pooled',
    n_bootstrap=500, 
    show_params=True, show_gof=True,
    suptitle=f'{ANIMAL_ID} - Sessions {idx_range[0]}-{idx_range[1]}')
plt.show()


In [ ]:
# =============================================================================
# Per-session fits overlaid (shows session-to-session variability)
# =============================================================================
idx_range = (5,10)
subset = animal.get_sessions(stage='Full_Task_Cont', idx_range=idx_range)

# all sessions
# subset = sessions

fig, infos = plot_session_psychometrics(
    subset, mode='overlay',
    suptitle=f'{ANIMAL_ID} — per-session psychometrics',
)
plt.show()


---

## 5. Single-Session Visualisation

Pick a session to inspect in detail.

In [ ]:
# =============================================================================
# Pick session
# =============================================================================

# Pick animal
animal = experiment.get_animal(aid)

SESSION_IDX = -1  # <-- change this (0-indexed, or -1 for last)
sess = animal.get_sessions(stage=STAGE, idx_range=SESSION_IDX)
arrays = sess.trials.get_model_arrays(exclude_abort=True, exclude_opto=True)
valid = ~arrays['no_response']

stim = arrays['stimuli'][valid]
choices = arrays['choices'][valid]
cats = arrays['categories'][valid]
correct = (choices == cats).astype(float)

print(f'Session: {sess.session_id}')
print(f'Date: {sess.date}')
print(f'Stage: {sess.metadata.stage}')
print(f'Trials: {len(stim)} valid / {sess.trials.n_trials} total')
print(f'Accuracy: {correct.mean():.1%}')
print(f'No-response rate: {arrays["no_response"].mean():.1%}')
print(f'Abort rate: {sess.trials.abort.mean():.1%}')


In [ ]:
# =============================================================================
# Panel 1: Psychometric curve
# =============================================================================

fig, ax = plt.subplots(figsize=(6, 5))
plot_psychometric(
    stim, choices, ax=ax, show_params=True, show_gof=True,
    title=f'{sess.session_id}',
)
plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# Panel 2: Trial-by-trial
# =============================================================================

fig = plot_session_trials(sess)
plt.show()

In [ ]:
# =============================================================================
# Panel 3: Reaction time distribution (if available)
# =============================================================================
data = sess
ignore_abort = False
# if arrays inputed
if isinstance(data, SessionData):
    rt = data.trials.reaction_time
    trial_idxs = data.trials.trial_number
    if ignore_abort:
        rt = rt[~data.trials.abort]
        trial_idxs = trial_idxs[~data.trials.abort]
        
elif isinstance(data, TrialData):
    rt = data.reaction_time
    trial_idxs = data.trial_number
    if ignore_abort:
        rt = rt[~data.abort]
        trial_idxs = trial_idxs[~data.abort]
elif isinstance(data, dict):
	rt = arrays['reaction_times']
	trial_idxs = arrays['trial_indices']
 

if len(rt) > 10:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Histogram
    axes[0].hist(rt, bins=40, color='steelblue', alpha=0.7, edgecolor='white')
    axes[0].axvline(np.median(rt), color='red', linewidth=1.5, linestyle='--',
                    label=f'Median: {np.median(rt):.0f} ms')
    axes[0].set_xlabel('Reaction time (ms)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('RT distribution')
    axes[0].legend()
    
    # RT over trials
    axes[1].scatter(trial_idxs, rt, s=10, alpha=0.5, c='steelblue')
    axes[1].set_xlabel('Trial')
    axes[1].set_ylabel('RT (ms)')
    axes[1].set_title('RT across session')
    
    plt.tight_layout()
    plt.show()
else:
    print(f'Not enough valid RTs ({len(rt)})')


---

## 6. Session Summary Stats

Full summary statistics for the selected session.

In [ ]:
# =============================================================================
# Compute and display all summary stats for this session
# =============================================================================

stats_dict = sess.compute_session_features()

pd.DataFrame([stats_dict])